# 🛠️ Agentic RAG with Google GenAI SDK

We can use the official `google-genai` SDK to build an agent that has access to tools (functions).

In [ ]:
import os
import nest_asyncio

nest_asyncio.apply()
os.environ['GEMINI_API_KEY'] = os.environ.get('GEMINI_API_KEY', 'YOUR_API_KEY')

In [ ]:
from google import genai
from google.genai import types
from pypdf import PdfReader

client = genai.Client()

# Define our tool as a standard Python function
def read_report_snippet(page_num: int) -> str:
    """Reads a specific page from the Apple Environmental Report."""
    try:
        reader = PdfReader('../data/Apple_Environmental_Progress_Report_2025.pdf')
        if page_num < len(reader.pages):
            return reader.pages[page_num].extract_text()
        return 'Page not found.'
    except Exception as e:
        return str(e)

In [ ]:
print('Initializing chat with tools...')
chat = client.chats.create(
    model='gemini-2.5-flash',
    config=types.GenerateContentConfig(
        tools=[read_report_snippet],
        temperature=0.0
    )
)

# The agent will decide to call `read_report_snippet`
response = chat.send_message('Can you read page 5 of the environmental report and summarize its contents?')
print(response.text)